In [1]:
from ucimlrepo import fetch_ucirepo

# Fetch dataset
auto_mpg = fetch_ucirepo(id=9)

# Features and target
X = auto_mpg.data.features
y = auto_mpg.data.targets

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (398, 7)
y shape: (398, 1)


In [2]:
X.head()

,displacement,cylinders,horsepower,weight,acceleration,model_year,origin
0,307.0,8,130.0,3504,12.0,70,1
1,350.0,8,165.0,3693,11.5,70,1
2,318.0,8,150.0,3436,11.0,70,1
3,304.0,8,150.0,3433,12.0,70,1
4,302.0,8,140.0,3449,10.5,70,1


In [3]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   displacement  398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   horsepower    392 non-null    float64
 3   weight        398 non-null    int64  
 4   acceleration  398 non-null    float64
 5   model_year    398 non-null    int64  
 6   origin        398 non-null    int64  
dtypes: float64(3), int64(4)
memory usage: 21.9 KB


In [4]:
X.isnull().sum()

displacement    0
cylinders       0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
dtype: int64

In [6]:
X.isnull().any().value_counts()

False    6
True     1
Name: count, dtype: int64

In [7]:
y = y.squeeze()

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (318, 7)
Test : (80, 7)


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

poly_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(
        degree=2,
        include_bias=False
    )),
    ("model", LinearRegression())
])

Raw X
  ↓
SimpleImputer
  ↓
Missing values replaced with median
  ↓
PolynomialFeatures
  ↓
x, x², interactions...
  ↓
LinearRegression
  ↓
MPG Prediction

In [10]:
poly_pipeline.fit(X_train,y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('poly', PolynomialFeatures(include_bias=False)),
                ('model', LinearRegression())])

In [11]:
y_train_pred = poly_pipeline.predict(X_train)
y_test_pred = poly_pipeline.predict(X_test)

In [12]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

import numpy as np

In [13]:
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

print("Train Performance")
print("MSE  :", train_mse)
print("RMSE :", train_rmse)
print("MAE  :", train_mae)
print("R²   :", train_r2)

Train Performance
MSE  : 6.762602722422855
RMSE : 2.600500475374472
MAE  : 1.9101504687269235
R²   : 0.8921369987356828


In [14]:
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("Test Performance")
print("MSE  :", test_mse)
print("RMSE :", test_rmse)
print("MAE  :", test_mae)
print("R²   :", test_r2)

Test Performance
MSE  : 5.955037914768425
RMSE : 2.440294636876544
MAE  : 1.8507544466381867
R²   : 0.8892424810080837


In [15]:
import pandas as pd

results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_test_pred
})

results.head(10)

,Actual,Predicted
198,33.0,32.328835
396,28.0,30.478663
33,19.0,19.631668
208,13.0,15.199615
93,14.0,13.335719
84,27.0,25.362656
373,24.0,27.415431
94,13.0,12.286923
222,17.0,16.638993
126,21.0,20.175810


In [16]:
results["Error"] = results["Actual"] - results["Predicted"]

results.head(10)

,Actual,Predicted,Error
198,33.0,32.328835,0.671165
396,28.0,30.478663,-2.478663
33,19.0,19.631668,-0.631668
208,13.0,15.199615,-2.199615
93,14.0,13.335719,0.664281
84,27.0,25.362656,1.637344
373,24.0,27.415431,-3.415431
94,13.0,12.286923,0.713077
222,17.0,16.638993,0.361007
126,21.0,20.175810,0.824190


## Key Takeaways

- Polynomial Regression helps capture nonlinear relationships.
- PolynomialFeatures creates powers and interaction features.
- Increasing degree increases model complexity.
- Very high degrees can cause overfitting and feature explosion.
- Train and test performance should both be evaluated.
- Pipeline simplifies preprocessing and prevents common mistakes.
- Final Auto MPG Test R²: 0.889
- Final Auto MPG Test MAE: 1.85